# Basic preparations: CAMELS-SPAT-MERIT-Hydro-Geofabric for Canada and Transboundary River Basins

In this Notebook, the geospatial fabric for the "Canada and transboundary river basin" is extracted from the `MERIT-Hydro-Basins` dataset.

Access to raw `MERIT-Basins` dataset is needed. Fir HPC, it is located in the following path:

```console
/project/6102189/data/misc-data/MERIT-Basins
```

The version used is `MERIT_Hydro_v07_Basins_v01_bugfix1` which is a directory under the root directory of the dataset.

Let's get started with our workflow and import necessary Python libraries:

In [1]:
# import geopandas as gpd # version 0.14.0
import pandas as pd # version 1.4.0
import numpy as np # version 1.22.2
import matplotlib.pyplot as plt # version 3.5.1
import geopandas as gpd # version 0.14.3
from shapely.geometry import Point # version 2.0.1

import hydrant.topology.geom as gm # version 0.1.0-dev1
import subprocess # built-in Python 3.10.2
import os # built-in Python 3.10.2
import glob # built-in Python 3.10.2

In [ ]:
###########################################################################################################

In [ ]:
# Part - 1: Final Merging Script (robust + counts + river-based Has_Gauge)
import os
import glob
import re
import geopandas as gpd
import pandas as pd 

# --- CONFIGURATION ---
root = "/project/6102189/data/misc-data/CAMELS-SPAT/shapefiles"
scales = ["headwater", "meso-scale", "macro-scale"]

cat_pattern = "*_distributed_basin.shp"
riv_pattern = "*_distributed_river.shp"

output_folder = "./merged_lamped"
os.makedirs(output_folder, exist_ok=True)

MISSING_INT = 1


# --- FUNCTIONS ---
def find_station_folders(scale):
    dist_path = os.path.join(root, scale, "shapes-distributed")
    if not os.path.exists(dist_path):
        return []
    return [f for f in glob.glob(os.path.join(dist_path, "*")) if os.path.isdir(f)]


def process_station(folder, scale):
    """Process one station (basin + river)."""
    cat_files = glob.glob(os.path.join(folder, cat_pattern))
    riv_files = glob.glob(os.path.join(folder, riv_pattern))

    if not cat_files:
        return None, None, 0, 0

    cat_path = cat_files[0]
    riv_path = riv_files[0] if riv_files else None

    cat_gdf = gpd.read_file(cat_path)
    if cat_gdf.empty:
        return None, None, 0, 0

    # Extract StationID
    filename = os.path.basename(cat_path)
    match = re.match(r"(USA|CAN)_(.*?)_distributed_basin\.shp", filename)
    station_id = match.group(2) if match else os.path.splitext(filename)[0]
    
    cat_gdf["StationID"] = station_id
    cat_gdf["Scale"] = scale
    cat_gdf["Has_Gauge"] = 0

    # --- CASE 1: single subbasin ---
    if len(cat_gdf) == 1:
        cat_gdf["Has_Gauge"] = 1

    # --- CASE 2: multiple subbasins → use river uparea ---
    else:
        if riv_path and os.path.exists(riv_path):
            riv_gdf = gpd.read_file(riv_path)

            if not riv_gdf.empty and "uparea" in riv_gdf.columns and "COMID" in riv_gdf.columns:
                riv_gdf["uparea"] = pd.to_numeric(riv_gdf["uparea"], errors="coerce").astype("float64")

                if riv_gdf["uparea"].notna().any():
                    idx_max = riv_gdf["uparea"].idxmax()
                    comid_outlet = riv_gdf.loc[idx_max, "COMID"]

                    if "COMID" in cat_gdf.columns:
                        mask = cat_gdf["COMID"] == comid_outlet
                        if mask.any():
                            cat_gdf.loc[mask, "Has_Gauge"] = 1
                        else:
                            cat_gdf.loc[cat_gdf.index[0], "Has_Gauge"] = 1
                    else:
                        cat_gdf.loc[cat_gdf.index[0], "Has_Gauge"] = 1
                else:
                    cat_gdf.loc[cat_gdf.index[0], "Has_Gauge"] = 1
            else:
                cat_gdf.loc[cat_gdf.index[0], "Has_Gauge"] = 1
        else:
            cat_gdf.loc[cat_gdf.index[0], "Has_Gauge"] = 1
    
    # ✅ ADD THIS LINE HERE (exact spot)
    cat_gdf.loc[cat_gdf["Has_Gauge"] != 1, "StationID"] = ""
    
    # --- River processing ---
    riv_gdf_out = None
    if riv_path and os.path.exists(riv_path):
        riv_gdf = gpd.read_file(riv_path)
        
        if not riv_gdf.empty:
            riv_gdf["StationID"] = station_id
            riv_gdf["Scale"] = scale
            
        # ---- ADD THIS ----
        numeric_cols = ["slope", "uparea", "maxup"]
        for col in numeric_cols:
            if col in riv_gdf.columns:
                riv_gdf[col] = pd.to_numeric(riv_gdf[col], errors="coerce")
                
        riv_gdf_out = riv_gdf

    return cat_gdf, riv_gdf_out, 1, (1 if riv_gdf_out is not None else 0)


def clean_gdf(gdf, is_river=False):
    if gdf is None or gdf.empty:
        return gdf

    if is_river:
        if "order" in gdf.columns:
            gdf["order"] = pd.to_numeric(gdf["order"], errors="coerce").fillna(MISSING_INT).astype("int64")
        else:
            gdf["order"] = MISSING_INT
        gdf = gdf[gdf.geometry.notna()].copy()

    for col in ["StationID", "Scale"]:
        if col in gdf.columns:
            gdf[col] = gdf[col].astype(str)

    return gdf


def safe_concat(gdf_list):
    """Concatenate GeoDataFrames safely (no warning, no crash)."""
    cleaned = []

    for gdf in gdf_list:
        if gdf is None or len(gdf) == 0:
            continue

        # Drop all-NA columns
        gdf = gdf.dropna(axis=1, how="all")

        # --- Ensure GeoDataFrame ---
        if not isinstance(gdf, gpd.GeoDataFrame):
            if "geometry" in gdf.columns:
                gdf = gpd.GeoDataFrame(gdf, geometry="geometry")
            else:
                continue  # skip if no geometry at all

        # Ensure valid geometry column
        if "geometry" not in gdf.columns:
            continue

        # Drop null geometries
        gdf = gdf[gdf.geometry.notna()]

        if gdf.empty:
            continue

        cleaned.append(gdf)

    if not cleaned:
        return gpd.GeoDataFrame()

    return gpd.GeoDataFrame(pd.concat(cleaned, ignore_index=True), geometry="geometry")


def save_shapefile(gdf, path, desc):
    if gdf is None or gdf.empty:
        print(f"   Skipped {desc} (empty): {path}")
        return
    gdf.to_file(path)
    print(f"   Saved {desc}: {path}")


# --- MAIN PROCESS ---
for scale in scales:
    print(f"\nProcessing scale: {scale}")

    folders = find_station_folders(scale)

    all_cat = []
    all_riv = []

    basin_files = 0
    river_files = 0
    missing_river = 0

    for folder in folders:
        cat_gdf, riv_gdf, bc, rc = process_station(folder, scale)

        basin_files += bc
        river_files += rc

        if bc == 1 and rc == 0:
            missing_river += 1

        if cat_gdf is not None:
            all_cat.append(cat_gdf)

        if riv_gdf is not None:
            all_riv.append(riv_gdf)

    # Merge safely
    cat_gdf = safe_concat(all_cat)
    riv_gdf = safe_concat(all_riv)

    # Clean
    cat_gdf = clean_gdf(cat_gdf, is_river=False)
    riv_gdf = clean_gdf(riv_gdf, is_river=True)

    # Feature counts
    n_subbasins = len(cat_gdf) if not cat_gdf.empty else 0
    n_rivers = len(riv_gdf) if not riv_gdf.empty else 0

    # Make outlets to have 'NextDownID' values of 0
    riv_gdf.loc[
        ~riv_gdf['NextDownID'].round(6).isin(riv_gdf['COMID'].round(6)),
        'NextDownID'
    ] = 0

    # --- SUMMARY ---
    print(f"  Basin shapefiles (stations) : {basin_files}")
    print(f"  River shapefiles (stations) : {river_files}")
    print(f"  Stations missing rivers     : {missing_river}")
    print(f"  Total subbasins (features)  : {n_subbasins}")
    print(f"  Total river reaches         : {n_rivers}")

    # Save
    print(f"Saving shapefiles for scale '{scale}':")
    save_shapefile(cat_gdf, os.path.join(output_folder, f"{scale}_basins.shp"), "Basins")
    save_shapefile(riv_gdf, os.path.join(output_folder, f"{scale}_rivers.shp"), "Rivers")

In [ ]:
# Part - 2: Robust hierarchical merge preserving Has_Gauge

import geopandas as gpd
import pandas as pd
import os

input_folder = "./merged_lamped"
output_folder = "./merged_lamped"
os.makedirs(output_folder, exist_ok=True)

# ------------------------
# Load function (no early deduplication)
# ------------------------
def load_gdf(path):
    gdf = gpd.read_file(path)
    print(f"Loaded {len(gdf)} features from {os.path.basename(path)}")
    # Ensure COMID is string for consistency
    if "COMID" in gdf.columns:
        gdf["COMID"] = gdf["COMID"].astype("float64")
    return gdf

# ------------------------
# Core hierarchical merge preserving Has_Gauge
# ------------------------
def merge_hierarchical_preserve_guage(head, meso, macro, name):
    print(f"\n--- Merging {name} ---")

    # Assign priority
    for gdf, p in zip([head, meso, macro], [1, 2, 3]):
        gdf["priority"] = p

    # Combine
    combined = pd.concat([head, meso, macro], ignore_index=True)

     # Preserve NextDownID: if COMID exists multiple times, take max
    if "NextDownID" in combined.columns:
        combined["NextDownID"] = combined.groupby("COMID")["NextDownID"].transform("max") 
    # Preserve order: if COMID exists multiple times, take max
    if "order" in combined.columns:
        combined["order"] = combined.groupby("COMID")["order"].transform("max")
    # Preserve Has_Gauge: if COMID exists multiple times, take max
    if "Has_Gauge" in combined.columns:
        combined["Has_Gauge"] = combined.groupby("COMID")["Has_Gauge"].transform("max")
    # Preserve StationID: keep first non-null per COMID
    if "StationID" in combined.columns:
        combined["StationID"] = combined.groupby("COMID")["StationID"].transform("first")

    # Sort by priority (headwater first)
    combined = combined.sort_values(["COMID", "priority"])

    # Drop duplicates by COMID, keep first (highest priority)
    merged = combined.drop_duplicates(subset="COMID", keep="first")

    # Clean up
    merged = merged.drop(columns="priority")

    print(f"Final {name}: {len(merged)} features")
    if "Has_Gauge" in merged.columns:
        print("Has_Gauge counts by Scale:")
        print(merged.groupby("Scale")["Has_Gauge"].sum())
        print(f"Total Has_Gauge: {merged['Has_Gauge'].sum()}")

    return gpd.GeoDataFrame(merged, geometry="geometry", crs=head.crs)

# ------------------------
# Load BASINS
# ------------------------
head_b = load_gdf(os.path.join(input_folder, "headwater_basins.shp"))
meso_b = load_gdf(os.path.join(input_folder, "meso-scale_basins.shp"))
macro_b = load_gdf(os.path.join(input_folder, "macro-scale_basins.shp"))

# ------------------------
# Load RIVERS
# ------------------------
head_r = load_gdf(os.path.join(input_folder, "headwater_rivers.shp"))
meso_r = load_gdf(os.path.join(input_folder, "meso-scale_rivers.shp"))
macro_r = load_gdf(os.path.join(input_folder, "macro-scale_rivers.shp"))

# ------------------------
# Merge
# ------------------------
merged_basins = merge_hierarchical_preserve_guage(head_b, meso_b, macro_b, "basins")
merged_rivers = merge_hierarchical_preserve_guage(head_r, meso_r, macro_r, "rivers")

# ------------------------
# Save outputs
# ------------------------
basin_out = os.path.join(output_folder, "all_scales_merged_basins.shp")
river_out = os.path.join(output_folder, "all_scales_merged_rivers.shp")

if not merged_basins.empty:
    merged_basins.to_file(basin_out)
    print(f"\nSaved basins → {basin_out}")

if not merged_rivers.empty:
    merged_rivers.to_file(river_out)
    print(f"Saved rivers → {river_out}")

# ------------------------
# SANITY CHECK
# ------------------------
def sanity_check(name, merged):
    print(f"\n=== CHECK: {name.upper()} ===")
    total = len(merged)
    unique_ids = merged["COMID"].nunique()
    print(f"Total features     : {total}")
    print(f"Unique COMIDs      : {unique_ids}")
    if total != unique_ids:
        print("❌ ERROR: duplicates still exist!")
    else:
        print("✅ No duplicates (correct)")
    if "Scale" in merged.columns:
        print("\nScale distribution:")
        print(merged["Scale"].value_counts())
    if "Has_Gauge" in merged.columns:
        print("\nTotal Has_Gauge:", merged["Has_Gauge"].sum())

sanity_check("basins", merged_basins)
sanity_check("rivers", merged_rivers)

In [ ]:
# Part 3: Correct missing river network informations: headwater without river networks add next_id
import geopandas as gpd
import pandas as pd
import numpy as np
import os

# ------------------------
# Paths
# ------------------------
basin_shp_path = "/home/zelalem/github-repos/community-workflows/1-geofabric/CAMELS-SPAT/merged_lamped/all_scales_merged_basins.shp"
river_shp_path = "/home/zelalem/github-repos/community-workflows/1-geofabric/CAMELS-SPAT/merged_lamped/all_scales_merged_rivers.shp"
output_path = "/home/zelalem/github-repos/community-workflows/1-geofabric/CAMELS-SPAT/merged_lamped/subbasins_missing_river.shp"

# ------------------------
# Load shapefiles
# ------------------------
inbasins = gpd.read_file(basin_shp_path)
inrivers = gpd.read_file(river_shp_path)
print(f"Loaded {len(inbasins)} basins and {len(inrivers)} river segments")

# Rename the river length
inrivers = inrivers.rename(columns={"new_len_km": "lengthkm"})

# ------------------------
# Ensure COMID / NextDownID are numeric
# ------------------------
for col in ["COMID", "NextDownID"]:
    if col in inbasins.columns:
        inbasins[col] = pd.to_numeric(inbasins[col], errors='coerce')
    if col in inrivers.columns:
        inrivers[col] = pd.to_numeric(inrivers[col], errors='coerce')

# Fix floating precision
inbasins["COMID"] = inbasins["COMID"].round(3)
inrivers["COMID"] = inrivers["COMID"].round(3)
inrivers["NextDownID"] = inrivers["NextDownID"].round(3)

# ------------------------
# Merge NextDownID into basins
# ------------------------
inbasins = inbasins.merge(
    inrivers[["COMID", "NextDownID"]],
    on="COMID",
    how="left"
)

# ------------------------
# Fill missing NextDownID 
# ------------------------
inbasins['NextDownID'] = inbasins['NextDownID'].fillna(0)

# ------------------------
# Optional: fix known flow direction issues manually
# ------------------------
fix_flow = {
    71017949.2: 71017949.1,
    71032880.2: 71032880.1,
    71037869.2: 71037869.1,
    72041796.2: 72041796.1,
    72044865.2: 72044865.1,
    72044865.3: 72044865.2,
    72044865.4: 72044865.3,
    72044865.5: 72044865.3,
    72044865.6: 72044865.5,
    72044865.7: 72044865.6,
    72044865.8: 72044865.7,
    72050055.2: 72050055.1,
    72055698.3: 72055698.1,
    73004375.2: 73004375.1,
    73004704.2: 73004704.1,
    78014375.2: 78014375.1,
    78023147.2: 78023147.1
}
for comid, downid in fix_flow.items():
    inbasins.loc[inbasins["COMID"] == comid, "NextDownID"] = downid

# ------------------------
# Verify outlets / headwaters
# ------------------------
n_outlets = (inbasins["NextDownID"] == 0).sum()
print(f"Total outlet headwater basins: {n_outlets}")

# ------------------------
# Identify subbasins without rivers
# ------------------------
mask_missing_rivers = ~inbasins["COMID"].isin(inrivers["COMID"])
orphan_basins = inbasins.loc[mask_missing_rivers].copy()
print(f"Subbasins missing rivers: {len(orphan_basins)}")

# ------------------------
# Save to shapefile
# ------------------------
# Remove existing files first to avoid shapefile conflicts
base = os.path.splitext(output_path)[0]
for ext in [".shp", ".shx", ".dbf", ".prj", ".cpg"]:
    f = base + ext
    if os.path.exists(f):
        os.remove(f)

orphan_basins.to_file(output_path)
print(f"Saved orphan subbasins → {output_path}")

In [ ]:
# Part 4: Updated function for generating river informations # improved version

import numpy as np
import geopandas as gpd
import rasterio
from rasterio.mask import mask as rio_mask
from rasterio.features import rasterize
from shapely.geometry import LineString
import pandas as pd


def derive_rivers_from_dem(
    basin_shp,
    flow_acc_tif,
    dem_tif,
    projected_crs="ESRI:102009",
    river_shp_out=None,
    debug=True
):
    # --------------------------
    # LOAD RASTERS
    # --------------------------
    fa_src = rasterio.open(flow_acc_tif)
    dem_src = rasterio.open(dem_tif)

    raster_crs = fa_src.crs

    # --------------------------
    # LOAD BASINS
    # --------------------------
    basins = gpd.read_file(basin_shp)
    basins = basins.to_crs(raster_crs)

    # --------------------------
    # TRACEBACK FUNCTION
    # --------------------------
    def trace_mainstem(fa, transform, threshold):
        """
        Returns:
          path_pixels: list of (row, col) indices along the mainstem
          line: LineString in raster CRS
        """

        # 1. Find the pixel with maximum FA inside the basin
        max_row, max_col = np.unravel_index(np.nanargmax(fa), fa.shape)


        # 2. Identify the downstream neighbor (highest FA among neighbors)
        neighbors = [(-1,-1), (-1,0), (-1,1),
                     (0,-1),         (0,1),
                     (1,-1),  (1,0), (1,1)]

        downstream_candidates = []
        for dr, dc in neighbors:
            r2 = max_row + dr
            c2 = max_col + dc
            if 0 <= r2 < fa.shape[0] and 0 <= c2 < fa.shape[1]:
                val = fa[r2, c2]
                if not np.isnan(val):
                    downstream_candidates.append((r2, c2, val))

        # First downstream cell (inside basin or boundary)
        if len(downstream_candidates) == 0:
            outlet_row, outlet_col = max_row, max_col
        else:
            outlet_row, outlet_col, _ = max(downstream_candidates, key=lambda x: x[2])

        # --------------------------------------------
        # NEW: add one more downstream step
        # --------------------------------------------
        # Move one more cell downstream using the same rule
        second_candidates = []
        for dr, dc in neighbors:
            r3 = outlet_row + dr
            c3 = outlet_col + dc
            if 0 <= r3 < fa.shape[0] and 0 <= c3 < fa.shape[1]:
                val = fa[r3, c3]
                if not np.isnan(val):
                    second_candidates.append((r3, c3, val))

        if len(second_candidates) > 0:
            # Replace outlet with the next downstream cell
            outlet_row, outlet_col, _ = max(second_candidates, key=lambda x: x[2])
        # --------------------------------------------            

        # 3. Start traceback from the downstream outlet
        row, col = outlet_row, outlet_col
        current_FA = fa[row, col]

        path_pixels = [(row, col)]

        # 4. Trace upstream (FA decreasing)
        while current_FA > threshold:
            candidates = []
            for dr, dc in neighbors:
                r2 = row + dr
                c2 = col + dc
                if 0 <= r2 < fa.shape[0] and 0 <= c2 < fa.shape[1]:
                    val = fa[r2, c2]
                    if not np.isnan(val) and val < current_FA:
                        candidates.append((r2, c2, val))

            if not candidates:
                break

            r2, c2, fa_val = max(candidates, key=lambda x: x[2])
            row, col = r2, c2
            current_FA = fa_val
            path_pixels.append((row, col))

        # 5. Convert pixel indices to coordinates
        if len(path_pixels) < 2:
            return [], LineString()

        rows = [p[0] for p in path_pixels]
        cols = [p[1] for p in path_pixels]

        xs, ys = rasterio.transform.xy(transform, rows, cols)
        line = LineString(zip(xs, ys))

        return path_pixels, line

    # --------------------------
    # MAIN LOOP
    # --------------------------
    all_rivers = []

    for idx, basin in basins.iterrows():
        comid = basin["COMID"]
        downcomid = basin["NextDownID"]
        updrarea = basin["unitarea"]
        geom = [basin.geometry]

        # CLIP FLOW ACCUMULATION
        fa_clip, fa_transform = rio_mask(fa_src, geom, crop=True)
        fa = fa_clip[0]

        # CLIP DEM
        dem_clip, dem_transform = rio_mask(dem_src, geom, crop=True)
        dem = dem_clip[0].astype(float)

        # Replace DEM NoData with NaN
        dem_nodata = dem_src.nodata
        if dem_nodata is not None:
            dem[dem == dem_nodata] = np.nan

        # STRICT MASK
        strict_mask = rasterize(
            [(basin.geometry, 1)],
            out_shape=fa.shape,
            transform=fa_transform,
            fill=0,
            all_touched=False
        )

        fa_strict = np.where(strict_mask == 1, fa, np.nan)

        if np.all(np.isnan(fa_strict)):
            continue

        max_fa = np.nanmax(fa_strict)
        threshold = max_fa / 2.0

        # TRACE MAIN STEM
        path_pixels, mainstem_raster_crs = trace_mainstem(fa_strict, fa_transform, threshold)

        if not path_pixels or mainstem_raster_crs.is_empty:
            continue

        # REPROJECT TO PROJECTED CRS FOR LENGTH/DISTANCE
        mainstem_proj = gpd.GeoSeries(
            [mainstem_raster_crs], crs=raster_crs
        ).to_crs(projected_crs).iloc[0]

        # ELEVATION SAMPLING
        rows = [p[0] for p in path_pixels]
        cols = [p[1] for p in path_pixels]
        elevations = np.array(dem[rows, cols], dtype=float)

        # --------------------------
        # LENGTH & SLOPE (TauDEM-like)
        # --------------------------
        # Length: use projected channel length
        length_m = float(mainstem_proj.length)

        # Slope: (z_upstream - z_downstream) / length
        valid = ~np.isnan(elevations)
        valid_elev = elevations[valid]

        if len(valid_elev) >= 2 and length_m > 0:
            # path_pixels[0] = downstream, path_pixels[-1] = upstream
            z_down = valid_elev[0]
            z_up = valid_elev[-1]
            dz = z_up - z_down
            slope = max(dz / length_m, 0.0)
        else:
            slope = 0.000001

        river_gdf = gpd.GeoDataFrame({
            "geometry": [mainstem_proj],
            "COMID": [comid],
            "NextDownID": [downcomid],
            "uparea": [updrarea],
            "threshold": [threshold],
            "lengthkm": [length_m],  # still meters here
            "slope": [slope],
            "source": ["generated"]
        }, crs=projected_crs)

        # Add required fields
        river_gdf["order"] = 1
        river_gdf["up1"] = 0
        river_gdf["up2"] = 0
        river_gdf["up3"] = 0
        river_gdf["up4"] = 0

        all_rivers.append(river_gdf)

    # MERGE RESULTS
    final_rivers = gpd.GeoDataFrame(
        pd.concat(all_rivers, ignore_index=True),
        crs=projected_crs
    )

    if river_shp_out:
        final_rivers["lengthkm"] = final_rivers["lengthkm"] / 1000.0
        final_rivers.to_file(river_shp_out)

    return final_rivers

In [ ]:
# Run dominant‑river extraction
missing_river_basin_path = "/home/zelalem/github-repos/community-workflows/1-geofabric/CAMELS-SPAT/merged_lamped/subbasins_missing_river.shp"
flow_acc_path = "/scratch/zelalem/cantrans-models/gistool-outputs/merit_hydro/CanTrans_model_upg.tif"
dem_path = "/scratch/zelalem/cantrans-models/gistool-outputs/merit_hydro/CanTrans_model_elv.tif"
target_crs_epsg = "ESRI:102009"
river_shp_path = "/home/zelalem/github-repos/community-workflows/1-geofabric/CAMELS-SPAT/merged_lamped/all_scales_merged_rivers_plus_generated.shp"

# Call the function 
new_rivers = derive_rivers_from_dem(
    basin_shp=missing_river_basin_path,
    flow_acc_tif=flow_acc_path,
    dem_tif=dem_path,
    projected_crs=target_crs_epsg,
    river_shp_out=river_shp_path,
    debug=False
)

# Copy the shapefile
rivers_existing = inrivers.copy()
rivers_generated = new_rivers.copy()

# Align CRS
if rivers_existing.crs != rivers_generated.crs:
    rivers_generated = rivers_generated.to_crs(rivers_existing.crs)
#
merged_rivers = gpd.GeoDataFrame(
    pd.concat(
        [
            rivers_existing[~rivers_existing["COMID"].isin(rivers_generated["COMID"])],
            rivers_generated
        ],
        ignore_index=True
    ),
    crs=rivers_existing.crs
)
merged_rivers.to_file(river_shp_path)

# Pre-process `CAMELS-SPAT-MERIT-Basins` Geofabric

### Subbasin aggregation function 

In [2]:
# Final Final # Exclude aggregation with intermediate with upstream gauge basin
# Basin Aggregation Function in Case of Gauge, Lakes and Reservoirs
import os
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Polygon, MultiPolygon

def remove_holes(geom):
    if geom is None:
        return None
    if geom.geom_type == "Polygon":
        return Polygon(geom.exterior)
    elif geom.geom_type == "MultiPolygon":
        return MultiPolygon([Polygon(p.exterior) for p in geom.geoms])
    return geom

def basin_aggregation(input_basin, input_river, min_SubArea, min_RivSlope, min_RivLength):
    """
    Aggregates basins and rivers based on drainage area, slope, and reservoir masking.

    Parameters:
    - input_basin (GeoDataFrame): Sub-basin geometries with attributes.
    - input_river (GeoDataFrame): River geometries with attributes.
    - min_SubArea (float): Minimum sub-basin area threshold.
    - min_RivSlope (float): Minimum river slope threshold.
    - min_RivLength (float): Minimum river length threshold.

    Returns:
    - agg_basin (GeoDataFrame): Aggregated basins.
    - agg_river (GeoDataFrame): Aggregated rivers.
    """

    # Add the columns from the river network that are not in the basin shapefile
    if 'COMID' not in input_basin.columns:
        raise ValueError("Missing 'COMID' in input_basin.")
        
    # Merge operation to join attribute table from input_river into input_basin
    missing_columns = [col for col in input_river.columns if col not in input_basin.columns[1:]]
    input_basin = input_basin.merge(input_river[missing_columns].copy(), on='COMID', how='left')   
    
    # Add flag and variable for future use
    if 'Lake_Cat' not in input_basin.columns:
        input_basin['Lake_Cat'] = 0
    if 'Has_Gauge' not in input_basin.columns:
        input_basin['Has_Gauge'] = 0
    if 'hillslope' not in input_river.columns:
        input_river['hillslope'] = 0 
        
    input_basin['Mask'] = 0
    input_basin.loc[input_basin['NextDownID'].fillna(-1) <= 0, 'Mask'] = 1
    input_basin.loc[input_basin['Has_Gauge'] > 0, 'Mask'] = 2
    input_basin.loc[input_basin['Lake_Cat'] > 0, 'Mask'] = 3
    input_basin['agg'] = input_basin['COMID']
    input_basin['aggdown'] = input_basin['NextDownID']

    # Initial filtering
    agg_basin = input_basin[['agg', 'aggdown', 'unitarea', 'uparea', 'Mask']].copy()
    agg_basin = agg_basin[~(
        ((agg_basin['aggdown'].fillna(-1) <= 0) & (agg_basin['uparea'] < min_SubArea)) | 
        (agg_basin['Mask'] == 3)
    )]
    lake_subs = input_basin[input_basin['Mask'] == 3]['agg']
    NoSubbasin = len(input_basin)

    while True:
        # Headwaters sub-basins Aggregation
        # Select the Headwater subbasins
        headwaters = (
            ~agg_basin['agg'].isin(agg_basin['aggdown']) &
            (agg_basin['unitarea'] < min_SubArea) &
            (agg_basin['Mask'] < 2)
        )
        small_subbasin = agg_basin[headwaters]
        small_subbasin = small_subbasin[~small_subbasin['aggdown'].isin(lake_subs)].sort_values(by='uparea', ascending=False)
        if not small_subbasin.empty:
            small_subbasin = small_subbasin.rename(columns={'agg': 'aggold', 'aggdown': 'agg'})
            xx = small_subbasin.merge(input_basin[['agg', 'aggdown']], on='agg', how='left')
            for ii in range(len(xx)):
                input_basin.loc[input_basin['agg'] == xx['aggold'].iloc[ii], 'aggdown'] = xx['aggdown'].iloc[ii]
                input_basin.loc[input_basin['agg'] == xx['aggold'].iloc[ii], 'agg'] = xx['agg'].iloc[ii]

            # ✅ FIXED: Aggregate all attributes in one step
            agg_basin = (
                input_basin
                .drop(columns=['geometry'])
                .groupby(['agg', 'aggdown'], as_index=False)
                .agg({
                    'unitarea': 'sum',       # Sum areas
                    'Mask': 'max',           # Keep if ANY is protected (0,1,2,3)
                    'uparea': 'max',         # Keep maximum drainage area
                    'Has_Gauge': 'max',      # Keep if ANY is gauged
                    'StationID' : 'first'
                })
            )
            
            agg_basin = agg_basin[~(
                ((agg_basin['aggdown'].fillna(-1) <= 0) & (agg_basin['uparea'] < min_SubArea)) | 
                (agg_basin['Mask'] == 3) 
            )]


        # Intermediate sub-basins Aggregation
        condition = (
            (
                agg_basin['agg'].isin(agg_basin['aggdown']) &
                (agg_basin['unitarea'] < min_SubArea) & 
                (agg_basin['Mask'] != 3)
            ) 
            | 
            (agg_basin['unitarea'] < 1.0)
        )


        small_subbasin = agg_basin[condition].sort_values(by='uparea', ascending=False)
        if not small_subbasin.empty:
            for jj in range(len(small_subbasin)):
                matches = input_basin[input_basin['COMID'] == small_subbasin['agg'].iloc[jj]]
                if matches.empty:
                    continue
                xx = matches.index[0]
                intermediate_agg = input_basin.loc[xx, 'agg']
                intermediate_unitarea = input_basin.loc[xx, 'unitarea']
                intermediate_aggdown = input_basin.loc[xx, 'aggdown']
                intermediate_mask = input_basin.loc[xx, 'Mask']
                
                # Check total area of all basins that has the same agg id value
                total_area = input_basin.loc[input_basin['agg'] == intermediate_agg, 'unitarea'].sum()
                
                if total_area < min_SubArea:
                    # Find ALL basins that drain into this intermediate basin
                    all_upstream = input_basin[input_basin['NextDownID'] == input_basin.loc[xx, 'COMID']]
                    
                    if not all_upstream.empty:
                        # Get the upstream basin with the LARGEST uparea
                        largest_upstream_idx = all_upstream['uparea'].idxmax()
                        
                        largest_upstream_mask = input_basin.loc[largest_upstream_idx, 'Mask']
                        largest_upstream_has_gauge = input_basin.loc[largest_upstream_idx, 'Has_Gauge']
                        largest_upstream_uparea = input_basin.loc[largest_upstream_idx, 'uparea']
                        largest_upstream_agg = input_basin.loc[largest_upstream_idx, 'agg']


                        # NEW LOGIC: Allow merge with gauge/lake only if this intermediate is < 1 km²
                        if largest_upstream_mask >= 2 and (total_area >= 1.0 or intermediate_mask == 2):
                            continue  # Protect gauge/lake unless intermediate is tiny
                        # Step 1: Identify basins that drain INTO the largest upstream
                        basins_flowing_into_upstream = input_basin[input_basin['aggdown'] == largest_upstream_agg]                          
                        # Step 2: Change largest upstream to intermediate's agg
                        input_basin.loc[input_basin['agg'] == largest_upstream_agg, 'agg'] = intermediate_agg
                        # Step 3: Update largest upstream basin's aggdown to intermediate's downstream
                        input_basin.loc[input_basin['agg'] == intermediate_agg, 'aggdown'] = intermediate_aggdown
                        # Step 4: UPDATE ONLY basins that drain INTO the largest upstream
                        if not basins_flowing_into_upstream.empty:
                            for _, flowing_basin in basins_flowing_into_upstream.iterrows():
                                input_basin.loc[input_basin['agg'] == flowing_basin['agg'], 'aggdown'] = intermediate_agg

            # ✅ FIXED: Aggregate all attributes in one step (consistent with headwater aggregation)
            agg_basin = (
                input_basin
                .drop(columns=['geometry'])
                .groupby(['agg', 'aggdown'], as_index=False)
                .agg({
                    'unitarea': 'sum',       # Sum areas
                    'uparea': 'max',         # Keep maximum drainage area
                    'Mask': 'max',           # Keep if ANY is protected (0,1,2,3)
                    'Has_Gauge': 'max',      # Keep if ANY is gauged
                    'StationID' : 'first'
                })
            )
            
            agg_basin = agg_basin[~(
                ((agg_basin['aggdown'].fillna(-1) <= 0) & (agg_basin['uparea'] < min_SubArea)) | 
                (agg_basin['Mask'] == 3)
            )]

        # Break if no small sub-basins are left
        if len(agg_basin[agg_basin['unitarea'] < min_SubArea]) == NoSubbasin:
            break
        NoSubbasin = len(agg_basin[agg_basin['unitarea'] < min_SubArea])
    
    # ------------------------------------------------------------------
    # Final aggregation of the sub-basins (WITH SLIVER REMOVAL)
    # ------------------------------------------------------------------
    # Dissolve - ✅ FIXED: Include all important attributes
    agg_basin = (
        input_basin
        .dissolve(
            by=['agg', 'aggdown'],
            aggfunc={
                'unitarea': 'sum',
                'uparea': 'max',
                'Mask': 'max',
                'Has_Gauge': 'max',
                'StationID' : 'first'
            },
            as_index=False
        )
        .rename(columns={
            'agg': 'COMID',
            'aggdown': 'NextDownID'
        })
    )

    # Fix invalid / hole / any artifacts from the input shapefile geometry
    agg_basin["geometry"] = agg_basin.geometry.buffer(0)
    agg_basin["geometry"] = agg_basin["geometry"].apply(remove_holes)
    agg_basin = agg_basin.explode(index_parts=False)
    agg_basin = agg_basin.dissolve(by="COMID")
    agg_basin = agg_basin.reset_index()
    
    # ------------------------------------------------------------------
    # Aggregating river network based on the aggregated sub-basins
    # ------------------------------------------------------------------
    agg_river = input_river.merge(
        input_basin[['COMID', 'agg', 'aggdown']],
        on='COMID',
        how='left'
    )
    # ------------------------------------------------------------------
    # Headwater detection (basins with no upstream basins)
    # ------------------------------------------------------------------
    upstream_ids = set(agg_basin['NextDownID'].dropna())
    headwater_ids = set(agg_basin['COMID']) - upstream_ids

    agg_river['mask'] = 0

    # ------------------------------------------------------------------
    # Dominant river tracing with headwater cutoff
    # ------------------------------------------------------------------
    for basin_id in agg_river['agg'].unique():
        xx = agg_river.index[agg_river['agg'] == basin_id].tolist()
        if not xx:
            continue
        headwater_uparea = agg_river.loc[xx, 'uparea'].max()
        is_headwater = basin_id in headwater_ids        

        while True:
            yy = agg_river.loc[xx, 'uparea'].idxmax()
            agg_river.at[yy, 'mask'] = 1
                
            xx = agg_river.index[
                agg_river['NextDownID'] == agg_river.at[yy, 'COMID']
            ].tolist()

            if len(xx) == 0:
                break

            yy_up = agg_river.loc[xx, 'uparea'].idxmax()
            if is_headwater and agg_river.at[yy_up, 'uparea'] < 0.5 * headwater_uparea:
                break
    
    agg_river = agg_river[agg_river['mask'] == 1].copy()

    # ------------------------------------------------------------------
    # Length-weighted slope aggregation
    # ------------------------------------------------------------------
    agg_river['slope'] = agg_river['slope'] * agg_river['lengthkm']
    agg_river = (
        agg_river
        .dissolve(
            by=['agg', 'aggdown'],
            aggfunc={
                'uparea': 'max',
                'lengthkm': 'sum',
                'slope': 'sum',
                'order': 'min',
                'hillslope': 'min',
                'StationID' : 'first'
            },
            as_index=False
        )
        .rename(columns={'agg': 'COMID',
                        'aggdown': 'NextDownID'})
    )

    agg_river['slope'] = (
        agg_river['slope'] /
        agg_river['lengthkm'].replace(0, np.nan)
    )

    # ------------------------------------------------------------------
    # Setting Minimum River attribute preprocessing ONLY for headwater basins
    # ------------------------------------------------------------------
    headwater_mask = agg_river['COMID'].isin(headwater_ids)
    agg_river.loc[headwater_mask, 'lengthkm'] = agg_river.loc[headwater_mask, 'lengthkm'].clip(lower=min_RivLength)
    agg_river.loc[headwater_mask, 'slope'] = agg_river.loc[headwater_mask, 'slope'].clip(lower=min_RivSlope)
    agg_river.loc[agg_river['lengthkm'] == 0, 'slope'] = np.nan
    
    # Return the aggregated basin and river shapefile
    return agg_basin, agg_river

In [3]:
# ==============================================================================
# USAGE EXAMPLE
# ==============================================================================

if __name__ == "__main__":
    
    # Define paths to the main directory and shapefiles to be aggregated
    dir_path = '/home/zelalem/github-repos/community-workflows/1-geofabric/CAMELS-SPAT/merged_lamped'
    input_basin = gpd.read_file(os.path.join(dir_path, 'all_scales_merged_basins.shp'))
    input_river = gpd.read_file(os.path.join(dir_path, 'all_scales_merged_rivers_plus_generated.shp'))
    
    # Define thresholds
    min_SubArea = 100           # km²
    min_RivSlope = 0.00001      # dimensionless
    min_RivLength = 1.0         # km
    
    print("Starting basin aggregation...")
    print(f"Input basins: {len(input_basin)}")
    print(f"Input rivers: {len(input_river)}")
    
    # Run aggregation
    agg_basin, agg_river = basin_aggregation(
        input_basin, 
        input_river, 
        min_SubArea=min_SubArea,
        min_RivSlope=min_RivSlope,
        min_RivLength=min_RivLength
    )
    
    # Save outputs
    agg_basin.to_file(os.path.join(dir_path, "agg_CAMEL-SPAT-MERIT_CanTrans_subbasins0.shp"))
    agg_river.to_file(os.path.join(dir_path, "agg_CAMEL-SPAT-MERIT_CanTrans_rivers0.shp"))
    
    print("\n" + "="*60)
    print("Aggregation complete!")
    print(f"Output basins: {len(agg_basin)} features")
    print(f"Output rivers: {len(agg_river)} features")
    print("="*60)

Starting basin aggregation...
Input basins: 30025
Input rivers: 30025

Aggregation complete!
Output basins: 8997 features
Output rivers: 8997 features


In [7]:
# Replace the decimal COMID by integer COMID
import geopandas as gpd

# Read shapefile
# Define paths to the main directory and shapefiles to be aggregated
dir_path = '/home/zelalem/github-repos/community-workflows/1-geofabric/CAMELS-SPAT/merged_lamped'
input_basin = gpd.read_file(os.path.join(dir_path, 'agg_CAMEL-SPAT-MERIT_CanTrans_subbasins0.shp'))
input_river = gpd.read_file(os.path.join(dir_path, 'agg_CAMEL-SPAT-MERIT_CanTrans_rivers0.shp'))

# Convert to string to safely manipulate
input_basin["COMID"] = input_basin["COMID"].astype(str)
input_river["COMID"] = input_river["COMID"].astype(str)
input_river["NextDownID"] = input_river["NextDownID"].astype(str)

# Function to replace decimal point with '0' if followed by in 
def fix_id(val):
    if val in ["nan", "None", ""]:
        return None
    
    val = str(val)
    
    if "." in val:
        base, dec = val.split(".")
        
        # If decimal part is zero → just return base
        if dec == "0":
            return base
        else:
            return base + "0" + dec
    
    return val
    
# Apply transformation
input_river["COMID_new"] = input_river["COMID"].apply(fix_id)
input_river["NextDownID_new"] = input_river["NextDownID"].apply(fix_id)

# Convert back to integer (handle missing values safely)
input_river["COMID_new"] = input_river["COMID_new"].astype(float).astype("Int64")
input_river["NextDownID_new"] = input_river["NextDownID_new"].astype(float).astype("Int64")

# Replace original columns
input_basin["COMID"] = input_basin["COMID"].map(
    input_river.set_index("COMID")["COMID_new"]
).fillna(input_basin["COMID"])

input_river["COMID"] = input_river["COMID_new"]
input_river["NextDownID"] = input_river["NextDownID_new"]

# Drop temp columns
input_river = input_river.drop(columns=["COMID_new", "NextDownID_new"])
input_basin = input_basin.drop(columns=["NextDownID"])

print("Done! Decimal points replaced with 0 and IDs remain unique.")

Done! Decimal points replaced with 0 and IDs remain unique.


## Subsetting Sub-basins and River segments for the Canada and Transboundary River Basins

In [9]:
# Read the study domain shapefile and prepare the Outlets
CanTrans = gpd.read_file('/home/zelalem/github-repos/community-workflows/1-geofabric/study-domain/CanTrans_MERIT_StudyDomain.shp')
CanTrans = CanTrans.rename(columns={'COMID': 'ID'})

# Ensure CRS match
if input_basin.crs is not None:
    # Reproject CanTrans to cat's CRS
    CanTrans = CanTrans.to_crs(input_basin.crs)
    
# Confirm the new CRS
# print(f'CanTrans CRS after projection: {CanTrans.crs}')

# Compute interior centroids
basins = input_basin.copy()
basins['interior_centroid'] = basins.geometry.apply(lambda geom: geom.representative_point())

# Create GeoDataFrame of centroids with COMID
centroid_gdf = gpd.GeoDataFrame(
    basins[['COMID']].copy(),
    geometry=basins['interior_centroid'],
    crs=basins.crs
)
# Spatial join: keep only centroids within CanTrans polygons
centroids_within_target = gpd.sjoin(centroid_gdf, CanTrans, predicate='within', how='inner')

# Extract unique COMIDs
matching_comids = centroids_within_target['COMID'].unique().tolist()

# Filter cat GeoDataFrame using matching_comids
filtered_cat = input_basin[input_basin['COMID'].isin(matching_comids)]
filtered_riv = input_river[input_river['COMID'].isin(matching_comids)]

# Save
filtered_cat.to_file(os.path.join(dir_path, "agg_CAMELS-MERIT_CanTrans_subbasins.shp"))
filtered_riv.to_file(os.path.join(dir_path, "agg_CAMELS-MERIT_CanTrans_rivers.shp"))

In [11]:
filtered_riv

,COMID,NextDownID,uparea,lengthkm,slope,order,hillslope,StationID,geometry
0,71001080,0,1886.982123,31.411296,0.000536,4,0,06FD002,"MULTILINESTRING ((-94.19792 58.01375, -94.1983..."
1,71002399,0,5888.184228,6.964702,0.000314,4,0,06FA001,"MULTILINESTRING ((-97.59792 57.31375, -97.5975..."
2,71002402,71002399,5606.331647,16.851223,0.000601,4,0,06FA001,"MULTILINESTRING ((-97.51833 57.29417, -97.5183..."
3,71002406,71002402,5013.128981,22.955224,0.000479,4,0,06FA001,"MULTILINESTRING ((-97.4825 57.17333, -97.48333..."
4,71002408,71002406,4844.644714,18.543687,0.000053,4,0,06FA001,"MULTILINESTRING ((-97.67917 57.1125, -97.68 57..."
...,...,...,...,...,...,...,...,...,...
8992,85008989,85008820,154.235565,17.805700,0.005312,2,0,10UH001,"LINESTRING (-69.55583 64.15833, -69.55667 64.1..."
8993,85009007,85008847,123.067798,15.714936,0.011288,2,0,10UH001,"LINESTRING (-68.7025 64.2175, -68.70167 64.218..."
8994,85009137,85008872,116.143679,14.928573,0.008704,1,0,10UH001,"LINESTRING (-68.67167 64.01167, -68.67083 64.0..."
8995,85009238,85008877,115.735660,16.849560,0.003763,1,0,10UH001,"LINESTRING (-69.07583 64.1675, -69.07583 64.16..."
